# 05 — Calibration of logit PDs: Bayesian vs Platt

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

MODEL = '_imputed_logit'
SOURCE_SETS = ['train']
TARGET_SETS = ['oot1', 'oot2']

In [ ]:
scores = pd.concat([pd.read_csv(f'../Model_scores/scores_{s}{MODEL}.csv').assign(set=s)
                    for s in SOURCE_SETS + TARGET_SETS], ignore_index=True)
scores['year'] = pd.to_datetime(scores['stmt_date']).dt.year
source = scores['set'].isin(SOURCE_SETS)
target = scores['set'].isin(TARGET_SETS)

pi_s = scores.loc[source, 'y_true'].mean()
pi_t = scores.loc[target, 'y_true'].mean()
print(f'pi_s = {pi_s:.4%}  pi_t = {pi_t:.4%}')

In [ ]:
def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

def bayes(p, pi_s, pi_t):
    return 1 / (1 + np.exp(-(logit(p) + logit(pi_t) - logit(pi_s))))

scores = scores[target].copy()

platt = LogisticRegression(C=1e10, max_iter=1000)
platt.fit(logit(scores[['y_pred']].to_numpy()), scores['y_true'])
a, b = platt.intercept_[0], platt.coef_[0, 0]
print(f'Platt: a = {a:.4f}, b = {b:.4f}')

scores['pd_raw'] = scores['y_pred']
scores['pd_bayes'] = bayes(scores['y_pred'], pi_s, pi_t)
scores['pd_platt'] = 1 / (1 + np.exp(-(a + b * logit(scores['y_pred']))))
PDS = ['pd_raw', 'pd_bayes', 'pd_platt']

In [ ]:
def log_likelihood(y, p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))

def check(g):
    y = g['y_true']
    out = {'firms': len(g), 'actual_defaults': y.sum()}
    for c in PDS:
        out[f'predicted_{c}'] = round(g[c].sum())
        out[f'LL_{c}'] = log_likelihood(y, g[c])
        out[f'AUC_{c}'] = roc_auc_score(y, g[c])
    for m in ['LL', 'AUC']:
        out[f'{m}_diff_bayes'] = out[f'{m}_pd_bayes'] - out[f'{m}_pd_raw']
        out[f'{m}_diff_platt'] = out[f'{m}_pd_platt'] - out[f'{m}_pd_raw']
    return pd.Series(out)

by_year = scores.groupby('year').apply(check, include_groups=False)
by_year.loc['total'] = check(scores)
print('Defaults'); display(by_year[['firms', 'actual_defaults'] + [f'predicted_{c}' for c in PDS]])
for name, m in [('Log-likelihood (higher = better)', 'LL'), ('AUC', 'AUC')]:
    print(name); display(by_year[[f'{m}_{c}' for c in PDS] + [f'{m}_diff_bayes', f'{m}_diff_platt']].round(4))

In [ ]:
def group_check(g, n=10):
    g = g.assign(group=pd.qcut(g['y_pred'].rank(method='first'), n, labels=False) + 1)
    t = g.groupby('group').agg(actual=('y_true', 'sum'), **{c: (c, 'sum') for c in PDS})
    for c in PDS:
        t[f'ratio_{c}'] = t['actual'] / t[c]
    return t.round(2)

for yr in sorted(scores['year'].unique()):
    print(yr); display(group_check(scores[scores['year'] == yr]))

In [ ]:
scores.to_csv(f'../Model_scores/scores_target{MODEL}_calibrated.csv', index=False)